# GIS to Unreal Engine 5.5 — Heightmap + Track Prep

## Dependencies

In [2]:
import os, glob, imageio, json
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.io import MemoryFile
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.fill import fillnodata
import matplotlib.pyplot as plt
import pyproj
import gpxpy
import imageio
from scipy.ndimage import zoom, gaussian_filter, distance_transform_edt, laplace
import contextily as ctx
from scipy.interpolate import RegularGridInterpolator

## DEM and GPX

### Methods

This cell takes raw USGS elevation tiles and a GPX track file and produces three outputs: a 16-bit heightmap PNG ready for direct import into Unreal Engine 5, a reprojected GeoTIFF of the DEM for use downstream, and a CSV of GPX track points converted into Unreal world coordinates for spline-based track rendering.

#### Why UTM

The source DEM tiles are in geographic coordinates (decimal degrees, EPSG:4326). Unreal Engine works in a Cartesian metric space where every unit is one centimeter. All spatial operations in this pipeline — pixel size, blur radius, noise scale, coordinate offsets — need to be in meters. The first thing the cell does is determine the correct UTM zone from the GPX track centroid and reproject everything to that zone. For the Wasatch, this is UTM Zone 12N (EPSG:32612). After reprojection, pixel sizes are in meters, distances are in meters, and the coordinate system is flat and Cartesian.

#### DEM Mosaicking

USGS 3DEP tiles are distributed as individual 1°×1° tiles. If the GPX track spans more than one tile, the cell merges them into a single in-memory mosaic before reprojection. The merge uses rasterio's `merge` function, which handles overlapping edges correctly. The mosaic lives in a `MemoryFile` to avoid writing intermediate files to disk.

#### UE5 Tile Size Constraints

Unreal Engine's landscape system requires heightmap dimensions to satisfy:

```
dimension = (n × components × quads_per_component) + 1
```

for integer values of those parameters. The `+1` arises because a grid of N quads requires N+1 vertices. Valid sizes include 505, 1009, 2017, 4033, and 8129 among others. This pipeline targets **4033×4033**, which gives (at ~27m pixels) a map extent of roughly 110km×110km — comfortably larger than the WURL track extent of ~17km×10km.

When multiple tiles are needed to cover the track, the cell calculates how many UE5 tiles are required along each axis using:

```python
stride = tile_size - 1   # 4032 — tiles share one edge row/column
n_tiles = ceil((track_extent - 1) / stride)
```

For this track, the result is 1×1: a single 4033×4033 tile covers the full extent. The crop window is centered on the GPX track centroid and aligned to exact tile boundaries.

#### Heightmap Normalization and UE5 Scale Parameters

Unreal Engine imports heightmaps as 16-bit grayscale PNGs where pixel value 0 maps to the minimum terrain height and pixel value 65535 maps to the maximum. The DEM is linearly rescaled to fill this range:

```python
dem_16 = ((elevation - dem_min) / (dem_max - dem_min) * 65535).astype(uint16)
```

This means the absolute elevation values are discarded — only relative height differences are preserved. The actual scale is recovered through two import parameters that the cell prints explicitly.

**Scale X/Y** sets how many centimeters each landscape unit (one pixel) represents in the horizontal plane:

```
Scale X/Y = pixel_size_m × 100
```

For this DEM at ~27.4m pixels: `Scale X/Y = 2742.71`.

**Scale Z** sets the vertical stretch. In UE5, a Scale Z of 100 maps the full 16-bit range to 512 × 100 = 51,200 cm of elevation. To match actual terrain relief, the cell back-solves for the Scale Z that makes the full 65535-step range equal to the actual elevation range:

```
Scale Z = (elevation_range_m × 100) / 512
```

For this DEM with a range of ~2302m: `Scale Z = 449.72`. These two values are printed prominently and must be entered manually during the UE5 landscape import dialog.

#### Bounds Metadata

The crop window's geographic bounds and pixel size are written to `heightmap_bounds.json`. This file is the alignment contract between the heightmap and all downstream rasters. The landcover cell reads it to ensure the landcover masks are reprojected to exactly the same grid — same origin, same pixel size, same dimensions — so that texture coordinates in Unreal map correctly onto the landscape.

#### GPX Track to Unreal Coordinates

The 940 GPX track points are converted from UTM meters into Unreal world space for use by a Blueprint that shrink-wraps the track spline down onto the landscape surface.

The conversion has four steps. First, the map center in UTM meters is computed from the crop window:

```python
map_center_easting  = crop_transform.x_origin + (image_width_m  / 2)
map_center_northing = crop_transform.y_origin - (image_height_m / 2)
```

Second, each track point is expressed as a signed offset from that center in meters. Third, the Y axis is negated to account for the difference between UTM convention (northing increases upward) and Unreal's coordinate system (Y increases downward in the horizontal plane). Fourth, the offsets are converted to centimeters by multiplying by 100, which matches the landscape Scale X/Y:

```python
x_unreal = offset_east_m  × 100
y_unreal = -offset_north_m × 100
```

Z is set to a fixed value well above the terrain (150,000 cm = 1500m). The Blueprint reads these points, places them as spline nodes, and projects each node downward onto the landscape surface at runtime. The CSV is exported to `track_points.csv` for import as a Data Table asset in UE5.

### Code

In [4]:
# --- PARAMETERS ---
input_paths = glob.glob("../data/wurl/usgs_tiles/*.tif")
gpx_path = "../data/wurl/WURL_Wasatch_Ultimate_Ridge_Linkup.gpx"
output_dir = "../data/wurl/unreal_output"
tile_size = 4033

os.makedirs(output_dir, exist_ok=True)

# --- 1. LOAD GPX & DETERMINE UTM ZONE ---
with open(gpx_path) as f:
    gpx = gpxpy.parse(f)
points = []
for track in gpx.tracks:
    for segment in track.segments:
        for pt in segment.points:
            points.append({'lat': pt.latitude, 'lon': pt.longitude, 'ele': pt.elevation or 0})
if not points:
    raise ValueError("No track points found in GPX.")
print(f"Loaded GPX: {len(points)} points")

center_lon = np.mean([p['lon'] for p in points])
center_lat = np.mean([p['lat'] for p in points])
zone = int(np.floor((center_lon + 180) / 6)) + 1
hemisphere = 6 if center_lat >= 0 else 7
target_crs = f"EPSG:326{zone:02d}" if hemisphere == 6 else f"EPSG:327{zone:02d}"
print(f"UTM zone: {zone}{'N' if hemisphere == 6 else 'S'}  ({target_crs})")

# Reproject GPX to UTM meters
proj = pyproj.Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
coords = np.array([proj.transform(p['lon'], p['lat']) for p in points])
elevs = np.array([p['ele'] for p in points])

E_min, E_max = coords[:, 0].min(), coords[:, 0].max()
N_min, N_max = coords[:, 1].min(), coords[:, 1].max()
print(f"Track bounds (UTM m): E [{E_min:.0f}, {E_max:.0f}], N [{N_min:.0f}, {N_max:.0f}]")

# --- 2. MOSAIC USGS TILES ---
opened = []
if len(input_paths) == 1:
    src = rasterio.open(input_paths[0])
    opened.append(src)
else:
    handles = [rasterio.open(p) for p in input_paths]
    mosaic_array, mosaic_transform = merge(handles)
    profile = handles[0].profile.copy()
    profile.update(height=mosaic_array.shape[1], width=mosaic_array.shape[2], transform=mosaic_transform)
    mem = MemoryFile()
    src = mem.open(**profile)
    src.write(mosaic_array)
    opened.extend(handles + [mem, src])

# --- 3. REPROJECT TO UTM ---
dst_crs = rasterio.crs.CRS.from_string(target_crs)
tfm, w_out, h_out = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)
p = src.profile.copy(); p.update(crs=dst_crs, transform=tfm, width=w_out, height=h_out)
m = MemoryFile(); src_r = m.open(**p)
for i in range(1, src.count + 1):
    reproject(rasterio.band(src, i), rasterio.band(src_r, i),
              src_transform=src.transform, src_crs=src.crs,
              dst_transform=tfm, dst_crs=dst_crs, resampling=Resampling.bilinear)
opened += [m, src_r]
print(f"Reprojected to {target_crs}: {w_out}x{h_out}")

# CHANGE 1: Capture pixel size after reprojection for reference
pixel_size_m = abs(src_r.transform.a)

# --- 4. FIND GPX CENTROID AND SET TILE-ALIGNED WINDOW ---
inv = ~src_r.transform
gpx_cols = inv.a * coords[:, 0] + inv.c
gpx_rows = inv.e * coords[:, 1] + inv.f
centroid_col = gpx_cols.mean()
centroid_row = gpx_rows.mean()

stride = tile_size - 1
cmin_gpx = gpx_cols.min()
cmax_gpx = gpx_cols.max()
rmin_gpx = gpx_rows.min()
rmax_gpx = gpx_rows.max()
n_tiles_x = max(1, int(np.ceil((cmax_gpx - cmin_gpx - 1) / stride)))
n_tiles_y = max(1, int(np.ceil((rmax_gpx - rmin_gpx - 1) / stride)))
target_cols = n_tiles_x * stride + 1
target_rows = n_tiles_y * stride + 1

cmin = int(round(centroid_col - (target_cols - 1) / 2))
rmin = int(round(centroid_row - (target_rows - 1) / 2))
cmax = cmin + target_cols
rmax = rmin + target_rows

cmin = max(0, cmin)
rmin = max(0, rmin)
cmax = min(w_out, cmax)
rmax = min(h_out, rmax)
target_cols = cmax - cmin
target_rows = rmax - rmin
n_tiles_x = int(np.ceil((target_cols - 1) / stride))
n_tiles_y = int(np.ceil((target_rows - 1) / stride))
print(f"Target distribution structure: {n_tiles_x}x{n_tiles_y} grid layouts")

# --- 6. CROP, FILL NODATA, NORMALIZE ---
w = Window(cmin, rmin, target_cols, target_rows)
cropped = src_r.read(window=w).astype(np.float32)
crop_tfm = rasterio.windows.transform(w, src_r.transform)

# Fill NoData voids
nodata = src_r.nodata
if nodata is not None:
    mask = cropped[0] != nodata
    cropped[0] = fillnodata(cropped[0], mask)

# Pad to exact grid math if window was truncated at DEM edge
rows, cols = cropped.shape[1], cropped.shape[2]
target_cols_exact = n_tiles_x * stride + 1
target_rows_exact = n_tiles_y * stride + 1
need_pad_c = max(0, target_cols_exact - cols)
need_pad_r = max(0, target_rows_exact - rows)
if need_pad_c > 0 or need_pad_r > 0:
    cropped = np.pad(cropped, ((0, 0), (0, need_pad_r), (0, need_pad_c)), mode='edge')
    rows, cols = cropped.shape[1], cropped.shape[2]

# Normalize to 16-bit
dem_min = float(cropped[0].min())
dem_max = float(cropped[0].max())
dem_16 = ((cropped[0] - dem_min) / (dem_max - dem_min) * 65535).astype(np.uint16)
print(f"Elevation range: {dem_min:.1f}m to {dem_max:.1f}m -> 0-65535")

# Z-Scale calculation for Unreal import
elevation_range = dem_max - dem_min
unreal_z_scale = (elevation_range * 100) / 512
print(f"--> WHEN IMPORTING TO UE5, SET THE PNG 'SCALE X/Y' TO: {100 * pixel_size_m:.4f}")
print(f"--> WHEN IMPORTING TO UE5, SET THE LANDSCAPE 'SCALE Z' TO: {unreal_z_scale:.4f}")

# --- 7. EXPORT HEIGHTMAP & FLUSH DRAG-AND-DROP TRACK ---

# 1. Export the heightmap
hm_dir = os.path.join(output_dir, "heightmaps")
os.makedirs(hm_dir, exist_ok=True)
hm_path = os.path.join(hm_dir, "Heightmap.png")
imageio.v3.imwrite(hm_path, dem_16, plugin="pillow")
print(f"Heightmap exported: {hm_path} ({cols}x{rows} px, {dem_16.nbytes / 1e6:.1f} MB)")

dem_tif_path = os.path.join(output_dir, "reprojected_dem.tif")
with rasterio.open(dem_tif_path, 'w', **src_r.profile) as dst:
    dst.write(src_r.read())
print(f"Reprojected DEM saved: {dem_tif_path}")

x_min = crop_tfm.c
x_max = x_min + cols * abs(crop_tfm.a)
y_max = crop_tfm.f
y_min = y_max - rows * abs(crop_tfm.e)
print(f"PNG bounds (UTM m): x_min={x_min:.1f} x_max={x_max:.1f} y_min={y_min:.1f} y_max={y_max:.1f}")
print(f"Pixel size: {pixel_size_m:.4f} m")

import json
bounds_meta = {
    "x_min": x_min, "y_min": y_min,
    "x_max": x_max, "y_max": y_max,
    "pixel_size_m": pixel_size_m
}
meta_path = os.path.join(output_dir, "heightmap_bounds.json")
with open(meta_path, 'w') as f:
    json.dump(bounds_meta, f, indent=2)
print(f"Bounds metadata saved: {meta_path}")

# 2. Capture the scale factor: How many real-world meters is 1 pixel?
pixel_size_m = abs(src_r.transform.a)

# 3. Find the geographic center of the image window in meters
image_width_meters = pixel_size_m * cols
image_height_meters = abs(src_r.transform.e) * rows

map_center_easting = crop_tfm.c + (image_width_meters / 2.0)
map_center_northing = crop_tfm.f - (image_height_meters / 2.0)

# 4. Calculate raw distance from the center in meters (Standard Cartesian coordinates)
x_meters_raw = coords[:, 0] - map_center_easting
y_meters_raw = coords[:, 1] - map_center_northing

y_meters_flipped = -y_meters_raw

# 6. COMPRESS SCALE TO MATCH UNREAL UNITS (Scale = pixel_size * 100 cm)
x_unreal = (x_meters_raw / pixel_size_m) * 100 * pixel_size_m
y_unreal = (y_meters_flipped / pixel_size_m) * 100 * pixel_size_m

# 7. FORCE CONSTANT Z FOR BLUEPRINT SHRINK WRAPPING
z_unreal = np.full_like(x_unreal, 150000.0)

# Export to your CSV Data Table
import pandas as pd
df = pd.DataFrame({
    'Point_ID': range(len(x_unreal)),
    'X': x_unreal,
    'Y': y_unreal,
    'Z': z_unreal
})

csv_path = os.path.join(output_dir, "track_points.csv")
df.to_csv(csv_path, index=False)
print(f"CSV successfully exported to: {csv_path}")

Loaded GPX: 940 points
UTM zone: 12N  (EPSG:32612)
Track bounds (UTM m): E [432505, 449341], N [4486021, 4496203]
Reprojected to EPSG:32612: 9483x12225
Target distribution structure: 1x1 grid layouts
Elevation range: 1265.3m to 3567.9m -> 0-65535
--> WHEN IMPORTING TO UE5, SET THE PNG 'SCALE X/Y' TO: 2742.7062
--> WHEN IMPORTING TO UE5, SET THE LANDSCAPE 'SCALE Z' TO: 449.7220
Heightmap exported: ../data/wurl/unreal_output\heightmaps\Heightmap.png (4033x4033 px, 32.5 MB)
Reprojected DEM saved: ../data/wurl/unreal_output\reprojected_dem.tif
PNG bounds (UTM m): x_min=385519.2 x_max=496132.5 y_min=4435829.0 y_max=4546442.3
Pixel size: 27.4271 m
Bounds metadata saved: ../data/wurl/unreal_output\heightmap_bounds.json
CSV successfully exported to: ../data/wurl/unreal_output\track_points.csv


## Landcover

### Method

Source data is the USGS GAP/LANDFIRE National Terrestrial Ecosystems 2011 dataset, accessed via Google Earth Engine at the link above using the bounds output from the DEM cell. Export at the heightmap resolution (4033×4033, ~27m pixel) in UTM Zone 12N to match the heightmap exactly.

#### Class Mapping

The raw GAP dataset contains dozens of cover types for northern Utah. Most can be collapsed into a small number of ecologically meaningful macro classes without meaningful loss of visual fidelity. The mapping used here is:

**Forest** (summed to a single macro class for material assignment, retained as subtypes for PCG)
- Aspen (148)
- Mixed Conifer (145, 138, 140)
- Pinyon-Juniper (183, 187, 186)
- Conifer (152, 156, 155, 151, 149, 158, 144, 153)
- Bigtooth Maple (154)
- Mountain Mahogany (184)

**Shrubland** (316, 312, 317)

**Steppe** (489, 491, 490, 498, 484, 485, 497, 488, 495)

**Grass / Alpine Meadow** (438, 503, 501, 315, 323, 325)

Forest receives six subtypes because individual tree species are large, visually distinct, and directly drive PCG foliage asset selection. Shrubland and steppe are generic enough that a single material handles each convincingly. Grass is separated from steppe because subalpine meadows are visually much richer than sagebrush steppe and warrant a distinct material.

Developed (581–584), Water (579), Cliff (529, 546), and Scree (549) are intentionally excluded from this pipeline. Cliff and scree are generated procedurally in Unreal from terrain slope and elevation derivatives. Developed and water features are handled separately as vector masks. The slope and elevation thresholds required to drive the Unreal-side cliff and scree materials are calculated here as a diagnostic output.

#### Processing Pipeline

The pipeline has six stages, executed in strict order. Each stage depends on the output of the previous one.

---

**Stage 1 — Purge and In-paint**

All pixels belonging to non-vegetative classes (developed, water, cliff, scree) are set to null. The remaining pixels represent a pre-human, idealized natural baseline. The null pixels are then filled using nearest-neighbor dilation: each null cell takes the value of its nearest non-null neighbor. This ensures the subsequent blur and softmax operate on a complete, hole-free canvas rather than propagating null values into adjacent vegetative regions.

---

**Stage 2 — One-hot Encoding and Gaussian Blur**

The 9-class label raster is one-hot encoded into 9 independent binary channels. Each channel is then blurred with a Gaussian kernel at `sigma = 200m` (approximately 7.3 pixels at 27m resolution). This dissolves the hard 30-meter grid edges of the satellite data into smooth, continuous gradients. Without this step, material boundaries in Unreal would appear as sharp staircase artifacts aligned to the source pixel grid.

---

**Stage 3 — Temperature-Scaled Softmax**

After blurring, a temperature-scaled softmax is applied across the 9-channel axis at every pixel:

$$\text{softmax}_i = \frac{e^{x_i / T}}{\sum_j e^{x_j / T}}$$

with temperature `T = 0.15`. The softmax enforces two properties simultaneously: all 9 channels sum to exactly 1.0 at every pixel, and the low temperature sharpens the distribution so that the dominant class at each pixel pushes toward 1.0 rather than producing flat, ambiguous probabilities. The result is a set of 9 continuous probability fields representing the likelihood of each cover type at each location, with smooth transitions at class boundaries.

The 9-channel softmax array is an intermediate product only. Nothing is exported at this stage.

---

**Stage 4 — Macro Class Collapse**

The 9 softmax channels are collapsed to 4 macro channels by summing the 6 forest subtype channels:
Forest    = Aspen + Mixed_Conifer + Pinyon_Juniper + Conifer + Bigtooth_Maple + Mountain_Mahogany

Shrubland = channels[6]

Steppe    = channels[7]

Grass     = channels[8]

Because the 9 softmax channels already sum to 1.0 per pixel, the 4 macro channels also sum to 1.0 per pixel automatically.

---

**Stage 5 — Dithered Binary Tournament**

The softmax macro channels are smooth probability fields: transition zones between classes contain intermediate values rather than a definitive winner. A naive binarization (threshold at 0.5) would produce hard, smooth boundaries. The goal instead is boundaries that are binary but spatially irregular — fingered and meandering the way real ecotones look on the ground.

This is achieved with three steps.

First, a transition-zone weight is computed at each pixel:
transition_w = (1.0 - max(macro channels)) ** 0.75

This weight is near zero in the interior of any dominant class (where one channel approaches 1.0) and near 1.0 at contested boundary pixels where no class clearly dominates. The 0.75 exponent slightly compresses the curve to keep mid-confidence pixels active.

Second, independent spatially-correlated fractal noise is generated for each macro channel. The noise is produced by summing multiple octaves of Gaussian-blurred white noise, with each octave at half the spatial scale and half the amplitude of the previous. The base scale is `sigma = 400m`, producing noise with a characteristic length scale roughly twice the blur sigma. This gives boundary irregularity that operates at ecologically plausible scales — large meanders at the kilometer scale with finer fingering at the hillslope scale.

Third, the noise is injected into each macro channel, scaled by the transition weight:
noisy[ch] = macro[ch] + fractal_noise * noise_amplitude * transition_w

A winner-takes-all argmax is then applied across the 4 noisy channels. The argmax guarantees that every pixel belongs to exactly one class. In the undisturbed interior of a dominant class, the noise is near-zero and the argmax outcome is unchanged. At the contested ecotone, the noise randomly tips the balance, producing an irregular binary boundary. The resulting 4 masks are exported as 8-bit grayscale PNGs where every pixel is either 0 or 255.

---

**Stage 6 — Within-Forest Subtype Renormalization**

The 6 forest subtype channels from the Stage 3 softmax are meaningful only inside the Forest macro winner pixels. Outside forest, they are zeroed. Inside forest, they are renormalized so that the 6 subtypes sum to 1.0 at every forest pixel:
subtype[ch] = (channels[ch] / sum(channels[0:6])) * forest_mask

This produces 6 probability fields that represent the relative likelihood of each tree species at a given forest location. These are not binary — they retain smooth gradients — because they are used by PCG as sampling weights for foliage asset placement, not as hard material boundaries.

---

#### Outputs

| File | Type | Use |
|------|------|-----|
| `Forest.png` | Binary 8-bit | Macro material mask |
| `Shrubland.png` | Binary 8-bit | Macro material mask |
| `Steppe.png` | Binary 8-bit | Macro material mask |
| `Grass.png` | Binary 8-bit | Macro material mask |
| `Aspen.png` | Probability 8-bit | PCG subtype weight (within forest only) |
| `Mixed_Conifer.png` | Probability 8-bit | PCG subtype weight (within forest only) |
| `Pinyon_Juniper.png` | Probability 8-bit | PCG subtype weight (within forest only) |
| `Conifer.png` | Probability 8-bit | PCG subtype weight (within forest only) |
| `Bigtooth_Maple.png` | Probability 8-bit | PCG subtype weight (within forest only) |
| `Mountain_Mahogany.png` | Probability 8-bit | PCG subtype weight (within forest only) |
| `concavity_mask.png` | Grayscale 8-bit | Chute and valley accent mask |

The 4 binary macro masks are guaranteed complementary: every pixel belongs to exactly one class and the four masks sum to 255 at every pixel. The 6 subtype probability masks sum to 1.0 at every forest pixel and are zero everywhere outside the Forest macro mask. Cliff and scree are handled in the Unreal material graph as procedural overrides layered on top of these masks using terrain slope and absolute elevation.

### Code

In [3]:
# --- Landcover: Macro Classes -> 9 Channel Softmaxed Probability Fields ---

lc_path = "../data/wurl/USGS_GAP_CONUS_2011_UTM12N_Export.tif"
dem_path = "../data/wurl/unreal_output/reprojected_dem.tif"
bounds_path = "../data/wurl/unreal_output/heightmap_bounds.json"
output_dir = "../data/wurl/unreal_output"
blur_sigma_m = 200.0
temperature = 0.15
noise_sigma_m = 400.0
noise_amplitude = 0.50
noise_seed = 42
n_octaves = 4

# --- 0. Load heightmap bounds & build target 4033x4033 grid ---
with open(bounds_path) as f:
    b = json.load(f)
x_min, y_min = b["x_min"], b["y_min"]
x_max, y_max = b["x_max"], b["y_max"]
ps = b["pixel_size_m"]
H_t, W_t = 4033, 4033
target_tfm = rasterio.Affine(ps, 0, x_min, 0, -ps, y_max)

print(f"Crop bounds: {x_min:.1f} {y_min:.1f} {x_max:.1f} {y_max:.1f}")
print(f"Pixel: {ps:.4f} m  ->  {W_t}x{H_t}")

# --- 1. Resample landcover (nearest) to heightmap grid ---
with rasterio.open(lc_path) as src:
    lc_full = src.read(1)
    tfm_lc = src.transform
    crs_lc = src.crs

lc = np.empty((H_t, W_t), dtype=np.uint16)
rasterio.warp.reproject(
    source=lc_full, destination=lc,
    src_transform=tfm_lc, src_crs=crs_lc,
    dst_transform=target_tfm, dst_crs=crs_lc,
    resampling=Resampling.nearest
)

# --- 2. Resample DEM (bilinear) to same grid ---
with rasterio.open(dem_path) as dem_src:
    dem_arr = dem_src.read(1).astype(np.float32)

dem = np.empty((H_t, W_t), dtype=np.float32)
rasterio.warp.reproject(
    source=dem_arr, destination=dem,
    src_transform=dem_src.transform, src_crs=dem_src.crs,
    dst_transform=target_tfm, dst_crs=crs_lc,
    resampling=Resampling.bilinear,
    src_nodata=dem_src.nodata, dst_nodata=-9999.0
)
valid_dem = dem > -9000
dem[~valid_dem] = np.nan

# --- 3. Slope ---
dzdx, dzdy = np.gradient(dem, ps, ps)
slope_deg = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))

# --- 4. Cliff (529,546) & Scree (549) stats ---
cliff_mask = np.isin(lc, [529, 546]) & valid_dem
scree_mask = (lc == 549) & valid_dem

if cliff_mask.any():
    print(f"Cliff min slope: {np.nanmin(slope_deg[cliff_mask]):.2f} deg")
if scree_mask.any():
    ms = np.nanmin(slope_deg[scree_mask])
    me = np.nanmin(dem[scree_mask])
    print(f"Scree min slope: {ms:.2f} deg")
    print(f"Scree min elevation: {me:.1f} m")
    dem_min_here = np.nanmin(dem[valid_dem])
    print(f"  -> Unreal Z: {(me - dem_min_here) * 100:.0f} cm")

# --- 5. Map raw GAP classes -> channel indices (0-9) ---
class_map = {
    148: 0,  # Aspen
    145: 1, 138: 1, 140: 1,  # Mixed Conifer
    183: 2, 187: 2, 186: 2,  # Pinyon-Juniper
    152: 3, 156: 3, 155: 3, 151: 3, 149: 3,
    158: 3, 144: 3, 153: 3,  # Conifer
    154: 4,  # Bigtooth Maple
    184: 5,  # Mountain Mahogany
    316: 6, 312: 6, 317: 6,  # Shrubland
    489: 7, 491: 7, 490: 7, 498: 7, 484: 7,
    485: 7, 497: 7, 488: 7, 495: 7,  # Steppe
    438: 8, 503: 8, 501: 8, 315: 8, 323: 8, 325: 8,  # Grass (Alpine Meadow + Subalpine)
}

class_idx = np.full(lc.shape, np.nan, dtype=np.float32)
for raw_val, ch in class_map.items():
    class_idx[lc == raw_val] = ch

# --- 5b. Slope histograms per subtype ---
ch_names = ["Aspen", "Mixed Conifer", "Pinyon-Juniper", "Conifer",
           "Bigtooth Maple", "Mountain Mahogany", "Shrubland", "Steppe",
           "Grass"]
fig, axes = plt.subplots(3, 3, figsize=(14, 16))
for ch in range(9):
    mask = (class_idx == ch) & valid_dem
    vals = slope_deg[mask]
    ax = axes[ch // 3, ch % 3]
    ax.hist(vals, bins=80, range=(0, 80), alpha=0.8, color=f"C{ch}")
    ax.axvline(np.nanmean(vals), color='k', linestyle='--',
               label=f"mean={np.nanmean(vals):.1f}°")
    ax.set_title(f"{ch}: {ch_names[ch]} (n={int(mask.sum()):,})")
    ax.set_xlabel("Slope (deg)")
    ax.set_ylabel("Pixel count")
    ax.legend(fontsize=8)
    ax.set_xlim(0, 80)
plt.tight_layout()
hist_dir = os.path.join(output_dir, "landcover")
os.makedirs(hist_dir, exist_ok=True)
hist_path = os.path.join(hist_dir, "slope_histograms.png")
plt.savefig(hist_path, dpi=150)
plt.close()
print(f"Slope histograms saved: {hist_path}")

# --- 6. In-paint holes (nearest-neighbor) -> clean int32 ---
hole = np.isnan(class_idx)
n_holes = hole.sum()
if n_holes and n_holes < hole.size:
    print(f"In-painting {n_holes} px ({n_holes / hole.size * 100:.1f}%)...")
    idx = distance_transform_edt(hole, return_distances=False, return_indices=True)
    class_idx = class_idx[tuple(idx)]
elif n_holes == hole.size:
    print("WARNING: all pixels are holes!")

class_idx = np.nan_to_num(class_idx, nan=0).astype(np.int32)

# --- 7. One-hot -> Blur -> Temp-scaled Softmax ---
channels = np.zeros((9, H_t, W_t), dtype=np.float32)
for ch in range(9):
    channels[ch] = (class_idx == ch).astype(np.float32)

blur_sigma_px = blur_sigma_m / ps
print(f"Blurring (sigma={blur_sigma_px:.1f} px = {blur_sigma_m:.0f} m)...")
for ch in range(9):
    channels[ch] = gaussian_filter(channels[ch], sigma=blur_sigma_px)

ch_max = channels.max(axis=0, keepdims=True)
e_x = np.exp((channels - ch_max) / temperature)
channels = e_x / e_x.sum(axis=0, keepdims=True)
print(f"Softmax applied (T={temperature}).")

lc_dir = os.path.join(output_dir, "landcover")
os.makedirs(lc_dir, exist_ok=True)

# --- 8. Collapse 9 softmax -> 4 macro channels ---
macro = np.stack([
    channels[0] + channels[1] + channels[2] + channels[3] + channels[4] + channels[5],
    channels[6], channels[7], channels[8]
], axis=0)
max_prob = macro.max(axis=0)
transition_w = (1.0 - max_prob) ** 0.75
macro_names = ["Forest", "Shrubland", "Steppe", "Grass"]

# --- 9. Fractal noise per macro channel ---
def _fractal_noise(shape, sigma_px, n_octaves, seed):
    rng = np.random.default_rng(seed)
    noise = np.zeros(shape, dtype=np.float32)
    for o in range(n_octaves):
        s = sigma_px / (2 ** o)
        a = 1.0 / (2 ** o)
        wn = rng.normal(size=shape).astype(np.float32)
        cn = gaussian_filter(wn, sigma=s)
        std = cn.std()
        if std > 0:
            cn = cn / std
        noise += cn * a
    std = noise.std()
    if std > 0:
        noise = noise / std
    return noise

noise_sigma_px = noise_sigma_m / ps
noisy = macro.copy()
for ch in range(4):
    nse = _fractal_noise((H_t, W_t), noise_sigma_px, n_octaves, noise_seed + ch * 1000)
    noisy[ch] = macro[ch] + nse * noise_amplitude * transition_w

winner = np.argmax(noisy, axis=0)
c0, c1, c2, c3 = ((winner == i).sum() for i in range(4))
print(f"Dither: Forest={c0}, Shrubland={c1}, Steppe={c2}, Grass={c3}")

# --- 10. Export 4 binary macro masks ---
for ch in range(4):
    binary = (winner == ch).astype(np.uint8) * 255
    path = os.path.join(lc_dir, f"{macro_names[ch]}.png")
    imageio.v3.imwrite(path, binary, plugin="pillow")
    print(f"  {macro_names[ch]}: {path}")

mask_sum = sum((winner == ch).astype(np.uint8) for ch in range(4))
assert (mask_sum == 1).all(), "Masks not complementary!"
print("  Complementarity verified.")

# --- 11. Renormalize 6 forest subtypes within forest mask ---
forest_mask = winner == 0
forest_sum = channels[0:6].sum(axis=0)
subtype_names = ["Aspen", "Mixed_Conifer", "Pinyon_Juniper", "Conifer",
                "Bigtooth_Maple", "Mountain_Mahogany"]
for ch in range(6):
    prob = (channels[ch] / (forest_sum + 1e-10)) * forest_mask
    arr = (prob * 255).clip(0, 255).astype(np.uint8)
    path = os.path.join(lc_dir, f"{subtype_names[ch]}.png")
    imageio.v3.imwrite(path, arr, plugin="pillow")
    print(f"  {subtype_names[ch]}: {path}")

# --- 12. Concavity mask (valleys/chutes) from DEM ---
dem_smooth = gaussian_filter(dem, sigma=6.0)
curvature = laplace(dem_smooth)
concavity = np.maximum(curvature, 0)
c_max = concavity.max()
if c_max > 0:
    concavity = concavity / c_max
concavity = np.power(concavity, 0.67)
concavity_8bit = (concavity * 255).clip(0, 255).astype(np.uint8)
concavity_path = os.path.join(lc_dir, "concavity_mask.png")
imageio.v3.imwrite(concavity_path, concavity_8bit, plugin="pillow")
print(f"  Concavity mask: {concavity_path}")

print("Done.")

Crop bounds: 385519.2 4435829.0 496132.5 4546442.3
Pixel: 27.4271 m  ->  4033x4033
Cliff min slope: 0.02 deg
Scree min slope: 0.22 deg
Scree min elevation: 2569.6 m
  -> Unreal Z: 130435 cm
Slope histograms saved: ../data/wurl/unreal_output\landcover\slope_histograms.png
In-painting 6438683 px (39.6%)...
Blurring (sigma=7.3 px = 200 m)...
Softmax applied (T=0.15).
Dither: Forest=5875245, Shrubland=3035179, Steppe=7057269, Grass=297396
  Forest: ../data/wurl/unreal_output\landcover\Forest.png
  Shrubland: ../data/wurl/unreal_output\landcover\Shrubland.png
  Steppe: ../data/wurl/unreal_output\landcover\Steppe.png
  Grass: ../data/wurl/unreal_output\landcover\Grass.png
  Complementarity verified.
  Aspen: ../data/wurl/unreal_output\landcover\Aspen.png
  Mixed_Conifer: ../data/wurl/unreal_output\landcover\Mixed_Conifer.png
  Pinyon_Juniper: ../data/wurl/unreal_output\landcover\Pinyon_Juniper.png
  Conifer: ../data/wurl/unreal_output\landcover\Conifer.png
  Bigtooth_Maple: ../data/wurl/unre